# Week 101, Day 1: Write a tutorial

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w101d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('tutorial.md', 'w', encoding='utf-8') as f:
    f.write('# Plan your first walk in Python\n\n## Goal\n\nBy the end you\'ll have a small program that finds the shortest walk between two places on a map.\n\n## Before you start\n\nYou need Python 3.10 or later and about twenty minutes. No libraries are needed.\n\n## Step 1: Describe the map\n\nEach path joins two places and has a length in kilometres. A dictionary of dictionaries holds them.\n\n```python\npaths = {"Mill": {"Weir": 1.5, "Church": 4.0}, "Weir": {"Mill": 1.5, "Church": 2.0}, "Church": {"Mill": 4.0, "Weir": 2.0}}\nprint(sorted(paths["Weir"]))\n```\n\n```text\n[\'Church\', \'Mill\']\n```\n\n## Step 2: Add up a route\n\nA route is a list of places; its length is the sum of the paths between neighbours.\n\n```python\ndef length(route):\n    return sum(paths[a][b] for a, b in zip(route, route[1:]))\n\nprint(length(["Mill", "Weir", "Church"]))\n```\n\n```text\n3.5\n```\n\n## Step 3: Find the shortest route\n\nDijkstra\'s algorithm always extends the shortest route found so far.\n\n```python\nimport heapq\n\ndef shortest(start, end):\n    heap = [(0, [start])]\n    seen = set()\n    while heap:\n        km, route = heapq.heappop(heap)\n        place = route[-1]\n        if place == end:\n            return km, route\n        if place in seen:\n            continue\n        seen.add(place)\n        for nxt, d in paths[place].items():\n            heapq.heappush(heap, (km + d, route + [nxt]))\n\nprint(shortest("Mill", "Church"))\n```\n\n```text\n(4.0, [\'Mill\', \'Church\'])\n```\n\n## What you built\n\nA map, a way to measure routes, and a shortest-route search: the heart of every route planner.\n')

**Predict before you run:** what will this print?



In [ ]:
import contextlib, io

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print("hidden from the screen")
print(repr(buf.getvalue()))

> **Key idea.** A tutorial takes a beginner to a working result: goal, prerequisites, small steps each with code and expected output, and a short summary. It must always work, so run it every time it changes. Introduce at most a few new names per step and show output after each. Explain each term the first time, prefer plain words, and watch a beginner follow it.

### Exercise: Reading the steps

Write `steps(markdown)`: split at lines starting with `## `; for each section whose title starts with `Step` (any case), return `(title, code, expected)`: the first fenced block marked `python` and the first marked `text` (each without its fences), or `None` where missing.

In [ ]:
import re

FENCE = "`" * 3

def steps(markdown):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_steps():\n    \"three steps\"\n    found = steps(open(\"tutorial.md\").read())\n    assert [t for t, _, _ in found] == [\"Step 1: Describe the map\", \"Step 2: Add up a route\", \"Step 3: Find the shortest route\"]\n    assert found[1][1].startswith(\"def length(route):\") and found[1][2] == \"3.5\"\n    md = \"## Intro\\n\\ntext\\n\\n## step 1: hello\\n\\n\" + FENCE + \"python\\nprint(1)\\n\" + FENCE + \"\\n\"\n    assert steps(md) == [(\"step 1: hello\", \"print(1)\", None)]")

### Exercise: Running the tutorial

Using `steps`, write `run_steps(markdown)`: run each step's code in order in **one shared namespace** (so later steps see earlier ones), capturing what it prints, and return `"TITLE: ok"` if the stripped output equals the stripped expected output, `"TITLE: output differs: GOT"` (GOT with `repr`), `"TITLE: no expected output"`, or `"TITLE: error: TYPE: MESSAGE"`.

In [ ]:
import contextlib
import io
import re

import contextlib, io, re
FENCE = "`" * 3
BLOCK = re.compile(FENCE + r"(\w+)\n(.*?)\n" + FENCE, re.S)

def steps(markdown):
    out = []
    for part in re.split(r"^## ", markdown, flags=re.M)[1:]:
        title = part.splitlines()[0].strip()
        if not title.lower().startswith("step"):
            continue
        blocks = BLOCK.findall(part)
        code = next((b for lang, b in blocks if lang == "python"), None)
        expected = next((b for lang, b in blocks if lang == "text"), None)
        out.append((title, code, expected))
    return out

def run_steps(markdown):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("FENCE = \"`\" * 3\n\ndef test_tutorial():\n    \"it found a mistake\"\n    assert run_steps(open(\"tutorial.md\").read()) == [\n        \"Step 1: Describe the map: ok\", \"Step 2: Add up a route: ok\",\n        \"Step 3: Find the shortest route: output differs: \\\"(3.5, ['Mill', 'Weir', 'Church'])\\\"\"]\n\ndef test_errors():\n    \"errors and missing output\"\n    md = (\"## Step 1: a\\n\\n\" + FENCE + \"python\\nx = 1\\n\" + FENCE + \"\\n\\n## Step 2: b\\n\\n\" + FENCE + \"python\\nprint(y)\\n\" + FENCE +\n          \"\\n\\n\" + FENCE + \"text\\n1\\n\" + FENCE + \"\\n\")\n    assert run_steps(md) == [\"Step 1: a: no expected output\", \"Step 2: b: error: NameError: name 'y' is not defined\"]")

### Exercise: The outline

Write `outline_problems(markdown)`: the section titles are the lines starting with `## `. Report, in order: `"missing: Goal"`, `"missing: Before you start"` and `"missing: What you built"` for absent sections (ignoring case); `"no steps"` if no title starts with `Step`; and `"steps not numbered 1 to N"` if the step titles' numbers (the first whole number in each) aren't exactly 1, 2, 3, ... in order.

In [ ]:
import re

def outline_problems(markdown):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_outline():\n    \"a tutorial's shape\"\n    assert outline_problems(open(\"tutorial.md\").read()) == []\n    assert outline_problems(\"## Goal\\n\\n## Step 1: a\\n\\n## Step 3: b\\n\") == [\"missing: Before you start\", \"missing: What you built\", \"steps not numbered 1 to N\"]\n    assert outline_problems(\"## Intro\\n\") == [\"missing: Goal\", \"missing: Before you start\", \"missing: What you built\", \"no steps\"]")

### Exercise: Too much at once?

Using `steps`, write `new_names(markdown, limit=3)`: for each step, the names its code **introduces** that no earlier step did: assigned names, function and class names, and imported names (found with `ast`; only top-level statements). Return `"TITLE: A, B"` (in the order they first appear, or `none`), adding `" (too many)"` when there are more than `limit`.

In [ ]:
import ast
import re

import contextlib, io, re
FENCE = "`" * 3
BLOCK = re.compile(FENCE + r"(\w+)\n(.*?)\n" + FENCE, re.S)

def steps(markdown):
    out = []
    for part in re.split(r"^## ", markdown, flags=re.M)[1:]:
        title = part.splitlines()[0].strip()
        if not title.lower().startswith("step"):
            continue
        blocks = BLOCK.findall(part)
        code = next((b for lang, b in blocks if lang == "python"), None)
        expected = next((b for lang, b in blocks if lang == "text"), None)
        out.append((title, code, expected))
    return out

def new_names(markdown, limit=3):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pacing():\n    \"a few new names per step\"\n    assert new_names(open(\"tutorial.md\").read()) == [\"Step 1: Describe the map: paths\", \"Step 2: Add up a route: length\",\n                                                     \"Step 3: Find the shortest route: heapq, shortest\"]\n    FENCE = \"`\" * 3\n    md = \"## Step 1: lots\\n\\n\" + FENCE + \"python\\nimport os, sys\\na, b = 1, 2\\ndef f():\\n    pass\\n\" + FENCE + \"\\n\"\n    assert new_names(md) == [\"Step 1: lots: os, sys, a, b, f (too many)\"]")

### Exercise: Terms explained before they're used

Write `unexplained(text, terms)`: return, in the order given, the terms that appear in the text (whole words, ignoring case) but never in a sentence that also contains ` is ` or ` means ` right after the term (for example `"A dictionary is ..."`). Sentences end at `.`, `!` or `?`.

In [ ]:
import re

def unexplained(text, terms):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_jargon():\n    \"say what it is\"\n    text = open(\"tutorial.md\").read()\n    assert unexplained(text, [\"dictionary\", \"heap\", \"algorithm\", \"route\"]) == [\"dictionary\", \"heap\", \"algorithm\"]\n    better = \"A dictionary is a set of key and value pairs. Keep the dictionary small.\"\n    assert unexplained(better, [\"dictionary\"]) == [] and unexplained(\"No terms here.\", [\"heap\"]) == []")

### Bug hunt: Output that escaped

This runs the step without capturing its output, so the text goes to the screen and the captured string is empty: it prints `hello` and then `''`. Capture it with `redirect_stdout` so it prints only `'hello'`.

In [ ]:
import contextlib, io

buf = io.StringIO()
exec("print('hello')", {})
print(repr(buf.getvalue().strip()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"redirect_stdout\"\n    expect_output(\"'hello'\")")

### Bug hunt: Steps that forgot each other

Each step runs in a fresh namespace, so step 2 can't see `paths` from step 1 and fails with a `NameError`. Share one namespace so this prints `3.5`.

In [ ]:
step1 = "paths = {'Mill': {'Weir': 1.5}, 'Weir': {'Church': 2.0}}"
step2 = "print(paths['Mill']['Weir'] + paths['Weir']['Church'])"
for code in (step1, step2):
    exec(code, {})

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one namespace\"\n    expect_output(\"3.5\")")

### Puzzle: Running one step

Arrange the body of `run_steps`' loop. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the Diátaxis kind of document a tutorial is (one word, describing what it's oriented to)
q = ___   # the most new names a step should usually introduce (an int)
r = ___   # True or False: a tutorial may skip showing output if the code is obvious
s = ___   # the context manager that captures print output (module.name)
t = ___   # the shortest Mill-to-Church route length on the tutorial's map

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p.lower() in (\"learning\", \"learning-oriented\") and q in (3, 4) and r is False, \"p, q and r\"\n    assert s == \"contextlib.redirect_stdout\" and t == 3.5, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: is the tutorial ready?

Write `tutorial_report(markdown, terms)` with today's helpers (given). Return `(lines, verdict)`: `"outline: ok"` or `"outline: "` and its problems joined by `"; "`; then each `run_steps` line; then each `new_names` line that ends in `(too many)`; then `"unexplained: A, B"` if `unexplained` finds any terms. `verdict` is `"ready to publish"` if the outline is ok, every step is ok, no step has too many names and nothing is unexplained; otherwise `"fix: "` and the failing kinds of `outline`, `steps`, `pacing` and `terms`, joined by `", "`.

In [ ]:
import ast
import contextlib
import io
import re

import contextlib, io, re
FENCE = "`" * 3
BLOCK = re.compile(FENCE + r"(\w+)\n(.*?)\n" + FENCE, re.S)

def steps(markdown):
    out = []
    for part in re.split(r"^## ", markdown, flags=re.M)[1:]:
        title = part.splitlines()[0].strip()
        if not title.lower().startswith("step"):
            continue
        blocks = BLOCK.findall(part)
        code = next((b for lang, b in blocks if lang == "python"), None)
        expected = next((b for lang, b in blocks if lang == "text"), None)
        out.append((title, code, expected))
    return out

def run_steps(markdown):
    ns, results = {}, []
    for title, code, expected in steps(markdown):
        buf = io.StringIO()
        try:
            with contextlib.redirect_stdout(buf):
                exec(code or "", ns)
            got = buf.getvalue().strip()
            status = "ok" if expected is not None and got == expected.strip() else f"output differs: {got!r}"
            if expected is None:
                status = "no expected output"
        except Exception as e:
            status = f"error: {type(e).__name__}: {e}"
        results.append(f"{title}: {status}")
    return results

def outline_problems(markdown):
    titles = [line[3:].strip() for line in markdown.splitlines() if line.startswith("## ")]
    lower = [t.lower() for t in titles]
    problems = [f"missing: {r}" for r in ("Goal", "Before you start", "What you built") if r.lower() not in lower]
    step_titles = [t for t in titles if t.lower().startswith("step")]
    if not step_titles:
        problems.append("no steps")
    elif [int(m.group()) if (m := re.search(r"\d+", t)) else None for t in step_titles] != list(range(1, len(step_titles) + 1)):
        problems.append("steps not numbered 1 to N")
    return problems

def new_names(markdown, limit=3):
    known, lines = set(), []
    for title, code, _ in steps(markdown):
        fresh = []
        for node in ast.parse(code or "").body:
            names = []
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
                names = [node.name]
            elif isinstance(node, (ast.Import, ast.ImportFrom)):
                names = [(a.asname or a.name).split(".")[0] for a in node.names]
            elif isinstance(node, ast.Assign):
                names = [n.id for t in node.targets for n in ast.walk(t) if isinstance(n, ast.Name)]
            fresh += [n for n in names if n not in known and n not in fresh]
        known.update(fresh)
        lines.append(f"{title}: " + (", ".join(fresh) or "none") + (" (too many)" if len(fresh) > limit else ""))
    return lines

def unexplained(text, terms):
    sentences = re.split(r"(?<=[.!?])\s+", " ".join(text.split()))
    out = []
    for term in terms:
        word = re.compile(rf"\b{re.escape(term)}s?\b", re.I)
        defining = re.compile(rf"\b{re.escape(term)}s?\s+(is|means)\s", re.I)
        if any(word.search(s) for s in sentences) and not any(defining.search(s) for s in sentences):
            out.append(term)
    return out

def tutorial_report(markdown, terms):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_report():\n    \"the page as it is\"\n    lines, verdict = tutorial_report(open(\"tutorial.md\").read(), [\"dictionary\", \"heap\"])\n    assert lines == [\"outline: ok\", \"Step 1: Describe the map: ok\", \"Step 2: Add up a route: ok\",\n                     \"Step 3: Find the shortest route: output differs: \\\"(3.5, ['Mill', 'Weir', 'Church'])\\\"\",\n                     \"unexplained: dictionary, heap\"]\n    assert verdict == \"fix: steps, terms\"\n\ndef test_fixed():\n    \"after the fixes\"\n    text = open(\"tutorial.md\").read().replace(\"(4.0, ['Mill', 'Church'])\", \"(3.5, ['Mill', 'Weir', 'Church'])\")\n    text = text.replace(\"A dictionary of dictionaries holds them.\", \"A dictionary is a collection of key and value pairs; one inside another holds them.\")\n    lines, verdict = tutorial_report(text, [\"dictionary\"])\n    assert verdict == \"ready to publish\" and lines[-1] == \"Step 3: Find the shortest route: ok\"")

## Warm-up

`redirect_stdout` sends `print` output into a string instead of the screen, so a program can check what another piece of code prints. That's how today's checker compares each tutorial step's output with the output the page promises.

## What a tutorial is

In Week 91's Diátaxis terms, a **tutorial** is learning-oriented: the reader is a beginner who wants to **do** something and come out able to do it again. It isn't a reference (no lists of every option) or an explanation (no history of the design). It has:

- a **goal** the reader can picture ("a program that finds the shortest walk");
- what they need **before they start**, honestly (version, time, prior knowledge);
- **steps**, each small, each ending with something that works and the output they'll see;
- a short **what you built** that names the achievement and points onwards.

A tutorial must **always work**: a beginner who follows it exactly and gets an error assumes it's their fault, and many give up. That's why every example should be run, every time the tutorial changes.

## Small steps

Each step adds one idea. A useful check: count the **new names** a step introduces (variables, functions, imports). More than three or four at once is a lot to hold. Show the output after every step, so readers can compare and catch their own mistakes early. Keep code blocks short (a dozen lines), and repeat context rather than make the reader scroll back.

## Words a beginner knows

The expert's curse is forgetting what it's like not to know. Every technical term needs either an explanation the first time it's used ("a **dictionary** is a collection of key and value pairs") or a link to one. Prefer plain words where they'll do ("list of places", not "sequence of vertices"). Then ask a beginner to follow it while you watch, as in Week 91's user testing: where they hesitate is where the tutorial needs work.

## Practice

## Quiz

## Daily challenge

## Your tutorial

1. **Choose a topic** you understand well and a beginner would want: something from your capstone, or from a week of this course you found hard at first.
2. **Write it** with the four sections and three to six small steps, every step with code and output.
3. **Run your checker** on it until every step is ok, and explain every term.
4. **Watch someone follow it** and note where they hesitate. Revise. Tomorrow you'll explain something aloud.

## Before you go

Tomorrow's teaching is spoken: a recorded explanation of one idea, with predictions for the viewer and answers to the misconceptions they'll bring.